In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

# Load the dataset
df = pd.read_csv('../data/cleaned_attendance_dataset.csv')

### 1. Drop or Transform 

In [2]:
# Parse Date
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Week_Number: weeks since 22-Jun-2026
start_date = pd.to_datetime('22-06-2026', format='%d-%m-%Y')
df['Week_Number'] = ((df['Date'] - start_date).dt.days // 7) + 1

# Day_Number_of_Semester
df['Day_Number_of_Semester'] = (df['Date'] - start_date).dt.days + 1

# select as the target vairable 
regression_target = df['Students_Present'].copy()

# Drop Students_Present
df = df.drop(columns=['Students_Present'])


# Compute thresholds from raw Attendance_Percentage
q33 = df['Attendance_Percentage'].quantile(0.33)
q67 = df['Attendance_Percentage'].quantile(0.67)

print(f"Low threshold : {q33:.2f}%")
print(f"High threshold: {q67:.2f}%")


# Convert Attendance_Percentage to a Band (Target for Classifier)
def assign_band(pct):
    if pct < q33:
        return 'Low'
    elif pct < q67:
        return 'Medium'
    else:
        return 'High'

# Assign bands using those thresholds
df['Attendance_Band'] = df['Attendance_Percentage'].apply(assign_band)

# Saving the thresholds immediately after:
joblib.dump({'low': q33, 'high': q67}, '../model/Random_Forest/band_thresholds.pkl')

# print output
print(df['Attendance_Band'].value_counts())
print()
print(df.groupby('Attendance_Band')['Attendance_Percentage'].describe().round(2))


# Compute Rolling_Avg_3 using Attendance_Percentage over Subject, Date, Lecture_Number
df = df.sort_values(by=['Subject', 'Date', 'Lecture_Number'])
df['Rolling_Avg_3'] = df.groupby('Subject')['Attendance_Percentage'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
df['Rolling_Avg_3'] = df['Rolling_Avg_3'].fillna(0)

# Convert Attendance_Percentage to a Band (Target for Classifier)
# df['Attendance_Band'] = pd.qcut(df['Attendance_Percentage'], q=3, labels=['Low', 'Medium', 'High'])

# Drop Attendance_Percentage and Date
df = df.drop(columns=['Attendance_Percentage', 'Date'])

# Dropping constant columns that are not useful as features
df = df.drop(columns=['End_Time', 'Semester', 'Branch', 'Section', 'Classroom', 'Total_Enrolled'], errors='ignore')


Low threshold : 13.73%
High threshold: 19.12%
Attendance_Band
High      176
Medium    175
Low       149
Name: count, dtype: int64

                 count   mean   std    min    25%    50%    75%    max
Attendance_Band                                                       
High             176.0  29.54  7.49  19.12  22.06  28.43  35.78  44.12
Low              149.0  11.49  1.36   7.84  10.78  11.76  12.75  13.24
Medium           175.0  15.82  1.56  13.73  14.22  15.69  17.16  18.63


## 2. Keep and Encode

In [3]:
from sklearn.preprocessing import LabelEncoder

# Separate LabelEncoders for each categorical column
le_subject = LabelEncoder()
le_faculty = LabelEncoder()
le_weather = LabelEncoder()

# Day_of_Week
day_map = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
df['Day_of_Week'] = df['Day_of_Week'].map(day_map)

# Start_Time: encode as hour (8, 9, 10, 11, 13 etc)
def extract_hour(time_str):
    time_str = time_str.strip()
    parts = time_str.split(' ')
    time_part = parts[0]
    meridian = parts[1] if len(parts) > 1 else ''
    
    # Safely handle both ':' and '.' as separator
    time_part = time_part.replace('.', ':')
    hour = int(time_part.split(':')[0])
    
    if meridian == 'PM' and hour != 12:
        hour += 12
    if meridian == 'AM' and hour == 12:
        hour = 0
    return hour

df['Start_Time'] = df['Start_Time'].apply(extract_hour)

# Subject, Faculty_ID, Weather
df['Subject'] = le_subject.fit_transform(df['Subject'])
df['Faculty_ID'] = le_faculty.fit_transform(df['Faculty_ID'])
df['Weather'] = le_weather.fit_transform(df['Weather'])

# Practical_Theory (Theory=0, Practical=1)
df['Practical_Theory'] = df['Practical_Theory'].map({'Theory': 0, 'Practical': 1})

# Binary encodes
binary_map = {'No': 0, 'Yes': 1}
df['Internal_Test_Week'] = df['Internal_Test_Week'].map(binary_map)
df['Assignment_Due'] = df['Assignment_Due'].map(binary_map)
df['Special_Event'] = df['Special_Event'].map(binary_map)

# Holiday_Before_After (No=0, Before=1, After=2)
holiday_map = {'No': 0, 'Before': 1, 'After': 2}
df['Holiday_Before_After'] = df['Holiday_Before_After'].map(holiday_map)

# Gap_Since_Previous_Lecture (Same Day=0, 1 Day=1, 2 Days=2, etc)
def parse_gap(gap_str):
    if gap_str == 'Same Day':
        return 0
    else:
        return int(gap_str.split(' ')[0])

df['Gap_Since_Previous_Lecture'] = df['Gap_Since_Previous_Lecture'].apply(parse_gap)


### 3. Engineer New Features 

In [4]:
# Is_First_Lecture_of_Day
df['Is_First_Lecture_of_Day'] = (df['Lecture_Number'] == 1).astype(int)

# Is_Afternoon (starts at 1.30 PM or later)
df['Is_Afternoon'] = (df['Start_Time'] >= 12).astype(int)

df = df.reset_index(drop=True)

df.head()


,Day_of_Week,Lecture_Number,Start_Time,Subject,Faculty_ID,Previous_Lecture_Attendance,Gap_Since_Previous_Lecture,Practical_Theory,Internal_Test_Week,Assignment_Due,Holiday_Before_After,Weather,Special_Event,Week_Number,Day_Number_of_Semester,Attendance_Band,Rolling_Avg_3,Is_First_Lecture_of_Day,Is_Afternoon
0,1,5,13,0,4,12,7,1,1,1,0,2,0,-9,-68,Medium,16.670000,0,1
1,1,5,13,0,4,34,3,1,0,0,0,1,0,-8,-61,Low,13.725000,0,1
2,1,5,13,0,4,22,7,1,1,1,0,2,0,-7,-54,Low,13.070000,0,1
3,1,5,13,0,4,24,2,1,0,0,0,2,0,-6,-47,High,14.213333,0,1
4,1,5,13,0,4,41,0,1,0,0,0,0,0,-5,-40,Medium,15.686667,0,1


#### 4. Train test split

In [7]:
# To perform a time-based split, we must sort the data purely chronologically.
from sklearn.model_selection import train_test_split
df = df.sort_values(by=['Day_Number_of_Semester', 'Start_Time'])

X = df.drop(columns=['Attendance_Band'])
y_clf= df['Attendance_Band']
y = regression_target

X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.2,random_state=42,stratify=y_clf)

# label_enc = LabelEncoder()
# y_train_enc = label_enc.fit_transform(y_train)
# y_test_enc  = label_enc.transform(y_test)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

print("Train band distribution:")
print(y_train.value_counts())
print()
print("Test band distribution:")
print(y_test.value_counts())
print()
print(f"Train size: {len(y_train)}")
print(f"Test size: {len(y_test)}")


Train size: 400, Test size: 100
Train band distribution:
Students_Present
31    22
27    20
29    20
23    18
25    16
      ..
67     1
51     1
43     1
44     1
63     1
Name: count, Length: 67, dtype: int64

Test band distribution:
Students_Present
28    7
22    6
33    5
25    5
20    5
26    5
31    4
38    4
27    4
29    4
35    3
34    3
39    3
50    2
68    2
30    2
32    2
58    2
70    2
80    2
79    2
40    1
67    1
82    1
72    1
21    1
60    1
53    1
65    1
85    1
76    1
48    1
23    1
62    1
69    1
42    1
37    1
43    1
24    1
52    1
17    1
57    1
77    1
55    1
78    1
47    1
16    1
Name: count, dtype: int64

Train size: 400
Test size: 100


## 5. Random Forest Regressor Training 

In [11]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [13]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.metrics import classification_report, accuracy_score

# We scale only the continuous features, while letting the binary/categorical features passthrough
cols_to_scale = ['Previous_Lecture_Attendance', 'Week_Number', 'Rolling_Avg_3', 'Gap_Since_Previous_Lecture', 'Day_Number_of_Semester']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_to_scale)
    ],
    remainder='passthrough'
)

# Initialize and train the random forest Classifier
g_Reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingRegressor(n_estimators=100, random_state=42,learning_rate=0.05, max_depth=4,min_samples_leaf=5 ))
])

g_Reg.fit(X_train, y_train)

# Predict and Evaluate
y_pred = g_Reg.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"MAE  : {mae:.2f} students")
print(f"RMSE : {rmse:.2f} students")
print(f"R²   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")


MAE  : 11.02 students
RMSE : 15.20 students
R²   : 0.3350
MAPE : 32.28%


In [ ]:
import joblib

joblib.dump(g_Reg, '../model/GradientBoosting/gradient-Boosting-regressor-model.pkl')


['../model/GradientBoosting/random_forest-regressor-model.pkl']